# Modèle des sorties du retard, pour corriger les codifications `PAY_1` = 1

**But** : produire les trois modèles qui servent à construire `PAY_1_corrigee`, la codification de septembre estimée pour les clients codifiés 1 (la codification 1 est lue comme un code d'attente : la banque n'a pas encore tranché, faute du paiement d'octobre). `PAY_1` d'origine est conservée à côté.

**Règle de construction de `PAY_1_corrigee`** :
1. `PAY_1` différent de 1 : `PAY_1_corrigee` = `PAY_1` ;
2. `PAY_1` = 1, **sain en août** (`PAY_2` <= 0) : `PAY_1_corrigee` = la codification d'août (un client sain en août le reste dans 95,8 % des cas ; l'entrée en retard n'est pas détectable avec ces données) ;
3. `PAY_1` = 1, **en retard en août** (`PAY_2` >= 2) : les trois modèles de ce notebook (MLP, CatBoost et SVM, trois familles différentes) estiment s'il **sort du retard**, chacun avec un seuil à 50 % de précision marginale ; sortie si **les trois l'annoncent** (unanimité, décision de l'utilisateur) → `PAY_1_corrigee` = -1 (dans les données, les sorties du retard sont codifiées -1 dans 99 % des cas) ; sinon retard maintenu → 2.

Avant la règle, les codifications 1 d'avril à août (`PAY_2` à `PAY_6`) sont corrigées par des règles fixes, des mois les plus anciens aux plus récents (section 2).

**Données** : `data/cleaned1_creditcard.csv` (niveau 1 du nettoyage : paiements géants et comptes inactifs retirés, codifications inchangées), **sans `dpnm`** (jamais lu), sans `ID` ni données personnelles dans les variables. La correction se fait **avant** le classement au contentieux.

**Ordre d'exécution** : `02_01_nettoyage.ipynb` produit `cleaned1`, puis ce notebook enregistre les modèles dans `src/modeles_pay1/` ; `02_01_nettoyage.ipynb` les recharge pour construire `PAY_1_corrigee` (section 6 : le code à reprendre). Les modèles ne sont à réentraîner que si les codifications d'avril à septembre du niveau 1 changent.

## 1. Données

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

df = pd.read_csv(root_dir / "data" / "cleaned1_creditcard.csv").drop(columns="dpnm")   # dpnm jamais utilisé
df = df.set_index("ID", drop=False)
PAY = [f"PAY_{n}" for n in range(1, 7)]
MOIS = {1: "septembre", 2: "août", 3: "juillet", 4: "juin", 5: "mai", 6: "avril"}
print(f"Clients du niveau 1 : {len(df)} ; PAY_1 = 1 : {int((df['PAY_1'] == 1).sum())} ; "
      f"codifications 1 d'avril à août : {int((df[PAY[1:]] == 1).sum().sum())}")

Clients du niveau 1 : 28851 ; PAY_1 = 1 : 3086 ; codifications 1 d'avril à août : 34


## 2. Correction des codifications 1 d'avril à août

Règles fixes, des mois les plus anciens (avril) aux plus récents (août) :
1. **solde du mois suivant nul ou créditeur** (`PAY_n` = 1 et `BILL_AMT(n-1)` <= 0) → la dernière codification saine qui précède ;
2. **encours nul** (`PAY_n` = 1, pas de facture du mois, `BILL_AMTn` <= 0, mois précédent sain) → la codification du mois précédent ;
3. **décisions client par client** pour les codifications 1 qui restent, prises en lisant la facture du mois et le paiement du mois suivant.

Après correction, il ne reste aucune codification 1 d'avril à août. `PAY_1` n'est pas touchée.

In [2]:
DECISIONS = {6783: {4: 0, 3: 0, 2: 0}, 11498: {4: 2, 3: 2, 2: 2}, 892: {3: 0, 2: 0}, 1967: {3: 0, 2: 0},
             5645: {2: 0}, 19195: {2: 2}, 25176: {2: 2}}   # ID : {n : nouvelle valeur de PAY_n}

def corriger_codifications_1(d):
    # Corrige les codifications 1 d'avril à août (PAY_6 à PAY_2) ; renvoie une copie et le nombre de codifications modifiées par règle
    d = d.copy()
    journal = {"solde du mois suivant nul": 0, "encours nul": 0, "décisions client par client": 0}
    for n in range(6, 1, -1):
        for k in d.index[(d[f"PAY_{n}"] == 1) & (d[f"BILL_AMT{n - 1}"] <= 0)]:
            saines = [d.at[k, f"PAY_{j}"] for j in range(n + 1, 7) if d.at[k, f"PAY_{j}"] <= 0]
            if saines:
                d.at[k, f"PAY_{n}"] = saines[0]; journal["solde du mois suivant nul"] += 1
    for n in range(5, 1, -1):
        regle = (d[f"PAY_{n}"] == 1) & (d[f"BILL_AMT{n}"] <= 0) & (d[f"PAY_{n + 1}"] <= 0)
        d.loc[regle, f"PAY_{n}"] = d.loc[regle, f"PAY_{n + 1}"]; journal["encours nul"] += int(regle.sum())
    for id_client, valeurs in DECISIONS.items():
        if id_client in d.index:
            for n, v in valeurs.items():
                if d.at[id_client, f"PAY_{n}"] == 1:
                    d.at[id_client, f"PAY_{n}"] = v; journal["décisions client par client"] += 1
    return d, journal

avant = df[PAY].copy()
df, journal = corriger_codifications_1(df)
assert int((df[PAY[1:]] == 1).sum().sum()) == 0, "il reste des codifications 1 d'avril à août"
assert (df["PAY_1"] == avant["PAY_1"]).all()
display(pd.Series(journal, name="Codifications modifiées").to_frame())
print("Par mois :", {MOIS[n]: int((df[f"PAY_{n}"] != avant[f"PAY_{n}"]).sum()) for n in range(6, 1, -1)})

,Codifications modifiées
solde du mois suivant nul,0
encours nul,21
décisions client par client,13


Par mois : {'avril': 0, 'mai': 0, 'juin': 2, 'juillet': 4, 'août': 28}


**Comment lire** : le nombre de codifications 1 corrigées par chaque règle, puis par mois. Une décision client par client ne s'applique que si la codification est encore un 1 après les deux règles.

## 3. Les exemples : clients en retard le mois précédent

Découpage de septembre : les clients dont `PAY_1` n'est pas un 1, 80/20 (`stratify` sur `PAY_1`, `random_state=42`) ; le test n'est lu qu'une fois, en section 5 bis, la règle étant figée. Exemples : les clients **en retard le mois précédent** (2 et plus), pour trois mois cibles : **septembre** (clients du train), **août** et **juillet** (clients du train et clients codifiés 1 en septembre, leur codification d'août ou de juillet étant tranchée). L'empilement s'arrête à juillet : à partir de juillet, la banque applique une pratique stable. Variables, rangées par ancienneté par rapport au mois cible : codifications des mois précédents (`codif_m1` = mois précédent, regroupées en -2, -1, 0, 2 et plus ; vide avant avril), soldes et paiements du mois (`m0`) et des mois précédents, plafond. Cible : 1 = **sortie du retard** (codification du mois -2, -1 ou 0), 0 = retard maintenu.

In [3]:
from sklearn.model_selection import train_test_split

connus, a_trancher = df[df["PAY_1"] != 1], df[df["PAY_1"] == 1]
ids_train, ids_test = train_test_split(connus.index, test_size=0.2, stratify=connus["PAY_1"], random_state=42)

def regrouper(codifications):
    # 2 et plus -> 2 (retard) ; -2, -1, 0 inchangés ; un mois absent (vide) reste vide
    return codifications.mask(codifications >= 2, 2)

def exemples_du_mois(d, n):
    # Ce que la banque voyait à la fin du mois n, en colonnes relatives (m1 = mois précédent) ; vide si le mois est antérieur à avril
    bloc = pd.DataFrame(index=d.index)
    for k in range(1, 6):
        bloc[f"codif_m{k}"] = regrouper(d[f"PAY_{n + k}"]).astype(float) if n + k <= 6 else np.nan
    for k in range(0, 6):
        bloc[f"solde_m{k}"] = d[f"BILL_AMT{n + k}"] if n + k <= 6 else np.nan
        bloc[f"paiement_m{k}"] = d[f"PAY_AMT{n + k}"] if n + k <= 6 else np.nan
    bloc["LIMIT_BAL"] = d["LIMIT_BAL"]
    bloc["codification_du_mois"] = d[f"PAY_{n}"]
    bloc["mois"] = n
    bloc["client"] = d.index
    return bloc

CATEGORIELLES = [f"codif_m{k}" for k in range(1, 6)]
NUMERIQUES = [f"solde_m{k}" for k in range(0, 6)] + [f"paiement_m{k}" for k in range(0, 6)] + ["LIMIT_BAL"]
VARIABLES = CATEGORIELLES + NUMERIQUES

emp = pd.concat([exemples_du_mois(df.loc[ids_train], 1)]
                + [exemples_du_mois(df.loc[ids_train.union(a_trancher.index)], n) for n in (2, 3)], ignore_index=True)
emp = emp[(emp["codif_m1"] == 2) & (emp["codification_du_mois"] != 1)].reset_index(drop=True)
emp["sortie"] = (emp["codification_du_mois"] <= 0).astype(int)
assert not emp["client"].isin(ids_test).any(), "un client du test est dans les exemples"
assert (emp["codif_m5"].notna() == (emp["mois"] == 1)).all()
X_emp, y_emp = emp[VARIABLES], emp["sortie"]
sept = np.flatnonzero(emp["mois"].to_numpy() == 1)

tableau = emp.groupby(emp["mois"].map(MOIS).rename("Mois cible")).agg(Exemples=("sortie", "size"), Sorties=("sortie", "sum"))
tableau["Part de sorties (%)"] = (tableau["Sorties"] / tableau["Exemples"] * 100).round(1)
display(tableau.reindex(["septembre", "août", "juillet"]))
print(f"Train de septembre : {len(ids_train)} clients ; test (non lu) : {len(ids_test)} ; PAY_1 = 1 à trancher : {len(a_trancher)}")

,Exemples,Sorties,Part de sorties (%)
Mois cible,,,
septembre,2035,344,16.9
août,3566,1059,29.7
juillet,2971,760,25.6


Train de septembre : 20612 clients ; test (non lu) : 5153 ; PAY_1 = 1 à trancher : 3086


**Comment lire** : une ligne par mois cible ; les exemples sont des clients en retard le mois précédent, dont une part sort du retard. Seule la ligne septembre sert à mesurer les modèles ; août et juillet ajoutent des exemples d'apprentissage.

## 4. Les trois modèles et leur mesure, pli par pli

**Modèles**, aux réglages retenus par l'étude de la codification 1 (boucle d'optimisation sur la précision moyenne des sorties, surapprentissage limité à 7 % relatif), trois familles différentes :
- **MLP** : moyenne des probabilités de 5 réseaux de graines différentes, couches (256, 128, 64), `alpha` 0,0001, arrêt anticipé ;
- **CatBoost** : profondeur 5, `learning_rate` 0,02, `l2_leaf_reg` 1, poids des classes `Balanced`, 1 000 itérations ;
- **SVM** à noyau gaussien : `C` 1, `gamma` 0,03, classes équilibrées ; meilleur des quatre candidats d'autres familles (régression logistique, SVM, KNN, bayésien naïf) dans l'étude. Il donne un score et non une probabilité : son seuil se calcule sur ce score de la même façon.

**Seuil de chaque modèle** : précision marginale de 50 % (décision de l'utilisateur) : les clients sont rangés du plus probablement sorti au moins probable, et on annonce des sorties tant que, parmi les 50 derniers annoncés, au moins 1 sur 2 est une vraie sortie (la sortie y est au moins aussi probable que le maintien). **Règle commune : sortie si les trois modèles l'annoncent.** Sur chaque pli, le seuil est fixé dans des plis internes, puis jugé sur le pli de validation (5 plis par client, notés sur septembre).

In [4]:
import time
from catboost import CatBoostClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVC

def log_montants(X):
    # Logarithme signé des soldes, des paiements et du plafond (un vide reste vide)
    X = X.copy()
    X[NUMERIQUES] = np.sign(X[NUMERIQUES]) * np.log1p(np.abs(X[NUMERIQUES]))
    return X

class MoyenneGraines(ClassifierMixin, BaseEstimator):
    # Moyenne des probabilités de plusieurs copies du même modèle, chacune avec sa graine
    def __init__(self, modele=None, graines=5, parametre_graine="mlpclassifier__random_state"):
        self.modele, self.graines, self.parametre_graine = modele, graines, parametre_graine

    def fit(self, X, y):
        self.modeles_ = [clone(self.modele).set_params(**{self.parametre_graine: g}).fit(X, y) for g in range(self.graines)]
        self.classes_ = self.modeles_[0].classes_
        return self

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.modeles_], axis=0)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

def score(modele, X):
    # Probabilité de sortie, ou score du SVM (qui ne donne pas de probabilité) : seul l'ordre des clients compte pour le seuil
    return modele.predict_proba(X)[:, 1] if hasattr(modele, "predict_proba") else modele.decision_function(X)

def preprocesseur_lineaire():
    # Codifications en une colonne par valeur ; montants (déjà au logarithme) : vide remplacé par 0 avec un indicateur, puis mise à l'échelle
    return ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
                              ("nombres", make_pipeline(SimpleImputer(strategy="constant", fill_value=0, add_indicator=True), StandardScaler()), NUMERIQUES)])

MODELES = {
    "MLP": MoyenneGraines(make_pipeline(
        FunctionTransformer(log_montants), preprocesseur_lineaire(),
        MLPClassifier(hidden_layer_sizes=(256, 128, 64), alpha=1e-4, max_iter=500, early_stopping=True, random_state=0))),
    "CatBoost": make_pipeline(
        FunctionTransformer(log_montants),
        ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES)], remainder="passthrough"),
        CatBoostClassifier(loss_function="Logloss", iterations=1000, depth=5, learning_rate=0.02, l2_leaf_reg=1, auto_class_weights="Balanced",
                           random_seed=42, verbose=0, thread_count=-1, allow_writing_files=False)),
    "SVM": make_pipeline(FunctionTransformer(log_montants), preprocesseur_lineaire(), SVC(kernel="rbf", C=1, gamma=0.03, class_weight="balanced")),
}
PRECISION_MARGINALE_VISEE = 0.50   # décision de l'utilisateur
VOIX_NECESSAIRES = 3                # sortie si les trois modèles l'annoncent (unanimité)
FENETRE_MARGE = 50

def seuil_marginal(y, proba, visee=PRECISION_MARGINALE_VISEE, fenetre=FENETRE_MARGE):
    # On annonce tant que la part de vraies sorties parmi les `fenetre` derniers annoncés reste >= visee ; probabilité du dernier annoncé
    ordre = np.argsort(-proba, kind="mergesort")
    y_tri, p_tri = np.asarray(y)[ordre], proba[ordre]
    if len(y_tri) < fenetre:
        return np.inf
    glissante = np.convolve(y_tri, np.ones(fenetre) / fenetre, mode="valid")
    sous = np.flatnonzero(glissante < visee - 1e-9)
    if len(sous) == 0:
        return p_tri[-1]
    return np.inf if sous[0] == 0 else p_tri[sous[0] + fenetre - 2]

def plis_par_client(indices):
    # 5 plis par client sur les exemples donnés, stratifiés sur les sorties de septembre, notés sur septembre
    sous = emp.iloc[indices]
    s = np.flatnonzero(sous["mois"].to_numpy() == 1)
    clients = sous["client"].to_numpy()
    plis = []
    for _, val in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(s, sous["sortie"].to_numpy()[s]):
        plis.append((indices[~np.isin(clients, np.unique(clients[s[val]]))], indices[s[val]]))
    return plis

plis = plis_par_client(np.arange(len(emp)))
y = y_emp.to_numpy()
probas, seuils, mesures = {}, {}, {}
for nom, modele in MODELES.items():
    debut = time.perf_counter()
    probas[nom], seuils[nom] = np.full(len(emp), np.nan), []
    mesures[nom] = {"Sorties annoncées": [], "Sorties retrouvées": [], "Précision des sorties annoncées": [], "Précision moyenne": []}
    for apprentissage, validation in plis:
        p_int, y_int = [], []
        for app_int, val_int in plis_par_client(apprentissage):
            p_int.append(score(clone(modele).fit(X_emp.iloc[app_int], y[app_int]), X_emp.iloc[val_int])); y_int.append(y[val_int])
        s = seuil_marginal(np.concatenate(y_int), np.concatenate(p_int))
        p = score(clone(modele).fit(X_emp.iloc[apprentissage], y[apprentissage]), X_emp.iloc[validation])
        probas[nom][validation] = p
        seuils[nom].append(s)
        annonce, y_v = p >= s, y[validation]
        mesures[nom]["Sorties annoncées"].append(int(annonce.sum()))
        mesures[nom]["Sorties retrouvées"].append(y_v[annonce].sum() / y_v.sum())
        mesures[nom]["Précision des sorties annoncées"].append(y_v[annonce].mean() if annonce.any() else np.nan)
        mesures[nom]["Précision moyenne"].append(average_precision_score(y_v, p))
    print(f"{nom} : fait ({round(time.perf_counter() - debut)} s)", flush=True)

def moy_et(v, facteur=100, decimales=1):
    v = np.asarray(v, dtype=float)
    return f"{np.nanmean(v) * facteur:.{decimales}f} ± {np.nanstd(v) * facteur:.{decimales}f}"

display(pd.DataFrame({nom: {"Seuil (min - max)": f"{min(seuils[nom]):.3f} - {max(seuils[nom]):.3f}",
                            "Sorties annoncées par pli": moy_et(m["Sorties annoncées"], 1),
                            "Sorties retrouvées (%)": moy_et(m["Sorties retrouvées"]),
                            "Précision des sorties annoncées (%)": moy_et(m["Précision des sorties annoncées"]),
                            "Précision moyenne sur les sorties (%)": moy_et(m["Précision moyenne"])} for nom, m in mesures.items()}).T)

# Règle retenue : sortie si les trois modèles l'annoncent (seuil de chaque pli)
annonce = {}
for nom in MODELES:
    a = np.zeros(len(emp), dtype=bool)
    for k, (_, validation) in enumerate(plis):
        a[validation] = probas[nom][validation] >= seuils[nom][k]
    annonce[nom] = a[sept]
y_sept = y[sept]
voix = sum(a.astype(int) for a in annonce.values())
regle = voix >= VOIX_NECESSAIRES
display(Markdown(f"**Règle retenue, les trois modèles d'accord** (septembre du train : {len(sept)} clients en retard en août, dont {int(y_sept.sum())} sorties) : "
                 f"{int(regle.sum())} sorties annoncées, dont {int(y_sept[regle].sum())} vraies ({y_sept[regle].mean() * 100:.1f} %) ; "
                 f"{y_sept[regle].sum() / y_sept.sum() * 100:.1f} % des sorties retrouvées"))

# Recoupement : quels modèles annoncent quels clients
noms = list(MODELES)
combinaison = pd.Series([" + ".join(n for n in noms if annonce[n][i]) or "aucun" for i in range(len(sept))])
recoupement = pd.DataFrame({"Clients": combinaison.value_counts(), "Vraies sorties": pd.Series(y_sept).groupby(combinaison).sum()})
recoupement["Part de vraies sorties (%)"] = (recoupement["Vraies sorties"] / recoupement["Clients"] * 100).round(1)
recoupement["Voix"] = [0 if k == "aucun" else k.count("+") + 1 for k in recoupement.index]
display(recoupement.sort_values(["Voix", "Clients"], ascending=False).rename_axis("Sortie annoncée par"))


MLP : fait (207 s)
CatBoost : fait (43 s)
SVM : fait (25 s)


,Seuil (min - max),Sorties annoncées par pli,Sorties retrouvées (%),Précision des sorties annoncées (%),Précision moyenne sur les sorties (%)
MLP,0.310 - 0.448,67.8 ± 4.2,81.4 ± 1.7,82.9 ± 4.7,87.3 ± 2.8
CatBoost,0.506 - 0.588,66.4 ± 3.3,81.1 ± 2.6,84.2 ± 3.6,88.2 ± 2.8
SVM,-0.134 - 0.024,71.0 ± 4.1,82.0 ± 1.9,79.6 ± 3.4,86.7 ± 2.2


**Règle retenue, les trois modèles d'accord** (septembre du train : 2035 clients en retard en août, dont 344 sorties) : 308 sorties annoncées, dont 272 vraies (88.3 %) ; 79.1 % des sorties retrouvées

,Clients,Vraies sorties,Part de vraies sorties (%),Voix
Sortie annoncée par,,,,
MLP + CatBoost + SVM,308,272,88.3,3
MLP + SVM,23,4,17.4,2
CatBoost + SVM,9,3,33.3,2
MLP + CatBoost,1,1,100.0,2
SVM,15,3,20.0,1
CatBoost,14,3,21.4,1
MLP,7,3,42.9,1
aucun,1658,55,3.3,0


**Comment lire** :
- **Sorties retrouvées** : sur 100 vraies sorties du retard, combien sont annoncées. **Précision des sorties annoncées** : sur 100 sorties annoncées, combien sont vraies. Avec une règle à la marge de 50 %, la précision de l'ensemble est plus haute que 50 %.
- **Règle retenue** : les sorties annoncées par les trois modèles ; leur part de vraies sorties est la précision des sorties estimées dans `PAY_1_corrigee`.
- **Recoupement** : une ligne par groupe de modèles qui annoncent la sortie (« aucun » : aucun modèle) ; seule la ligne à trois voix forme la règle.

## 5. Modèles finaux et enregistrement (`src/modeles_pay1/`)

Chaque modèle est appris sur **tous les exemples** (trois mois cibles) ; son seuil est fixé sur ses probabilités (ou scores, pour le SVM) hors pli de septembre (section 4). Les anciens fichiers de modèles du dossier sont retirés. Les modèles sont enregistrés avec `cloudpickle` (ils embarquent le logarithme et la moyenne des graines : ils se rechargent avec `pickle.load`, sans rien redéfinir), avec une synthèse (variables, seuils, nombre de voix nécessaires, versions des bibliothèques), avec une synthèse (variables, réglages, seuils, versions des bibliothèques).

In [5]:
import json, platform
from datetime import datetime
import catboost, sklearn
from joblib.externals import cloudpickle

dossier = root_dir / "src" / "modeles_pay1"
dossier.mkdir(parents=True, exist_ok=True)
for ancien in dossier.glob("sorties_retard_*.pkl"):
    ancien.unlink()   # seuls les modèles de la règle retenue restent dans le dossier
synthese = {"date": datetime.now().isoformat(timespec="seconds"),
            "source": "data/cleaned1_creditcard.csv, codifications 1 d'avril à août corrigées ; sans dpnm",
            "exemples": "clients en retard le mois précédent ; mois cibles septembre, août, juillet ; train de septembre seulement",
            "cible": "1 = sortie du retard (codification du mois -2, -1 ou 0) ; 0 = retard maintenu",
            "variables": VARIABLES,
            "regle_de_decision": f"précision marginale de {PRECISION_MARGINALE_VISEE:.0%} sur les {FENETRE_MARGE} derniers annoncés, pour chaque modèle ; sortie si les trois modèles l'annoncent",
            "voix_necessaires": VOIX_NECESSAIRES,
            "versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__, "catboost": catboost.__version__, "pandas": pd.__version__},
            "modeles": {}}
for nom, modele in MODELES.items():
    final = clone(modele).fit(X_emp, y)
    fichier = dossier / f"sorties_retard_{nom.lower()}.pkl"
    with open(fichier, "wb") as f:
        cloudpickle.dump(final, f)
    synthese["modeles"][nom] = {"fichier": fichier.name, "seuil": float(seuil_marginal(y_sept, probas[nom][sept])),
                                "score": "decision_function" if not hasattr(final, "predict_proba") else "predict_proba"}
    print(f"{nom} : {fichier.relative_to(root_dir)} ({fichier.stat().st_size // 1024} Ko), seuil {synthese['modeles'][nom]['seuil']:.3f}")
(dossier / "synthese.json").write_text(json.dumps(synthese, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"Synthèse : {(dossier / 'synthese.json').relative_to(root_dir)}")

MLP : src\modeles_pay1\sorties_retard_mlp.pkl (5997 Ko), seuil 0.368
CatBoost : src\modeles_pay1\sorties_retard_catboost.pkl (595 Ko), seuil 0.601
SVM : src\modeles_pay1\sorties_retard_svm.pkl (1396 Ko), seuil 0.070
Synthèse : src\modeles_pay1\synthese.json


## 5 bis. Vérification unique sur le test

La règle est figée (décision de l'utilisateur) : le test de septembre est lu **une seule fois pour cette règle** (une première lecture, le 10/10/2026, a jugé l'ancienne règle : deux modèles d'accord, seuils à 80 %). Les modèles finaux et leurs seuils (section 5) sont appliqués aux clients du test **en retard en août** ; sortie si les trois modèles l'annoncent. Puis la règle complète (sain en août → codification d'août ; en retard en août → sortie estimée -1 ou retard maintenu 2) est comparée à la vraie `PAY_1` de tous les clients du test, face à la règle « même codification qu'en août ». La cible est `PAY_1` ; `dpnm` n'est pas lu.

In [6]:
test = df.loc[ids_test]
retard_test = test["PAY_2"] >= 2
X_test = exemples_du_mois(test.loc[retard_test], 1)[VARIABLES]
sortie_vraie = (test.loc[retard_test, "PAY_1"] <= 0).to_numpy()
annonce_test = {}
for nom in MODELES:
    with open(dossier / synthese["modeles"][nom]["fichier"], "rb") as f:
        annonce_test[nom] = score(cloudpickle.load(f), X_test) >= synthese["modeles"][nom]["seuil"]
voix_test = sum(a.astype(int) for a in annonce_test.values())
regles = {f"{nom} seul": a for nom, a in annonce_test.items()}
regles["Les trois d'accord (règle retenue)"] = voix_test >= VOIX_NECESSAIRES
lignes = []
for libelle, a in regles.items():
    lignes.append({"Décision": libelle, "Sorties annoncées": int(a.sum()), "Vraies sorties parmi elles": int(sortie_vraie[a].sum()),
                   "Sorties retrouvées (%)": round(sortie_vraie[a].sum() / sortie_vraie.sum() * 100, 1),
                   "Précision des sorties annoncées (%)": round(sortie_vraie[a].mean() * 100, 1) if a.any() else np.nan})
display(Markdown(f"**Test : clients en retard en août** ({int(retard_test.sum())} clients, dont {int(sortie_vraie.sum())} sorties du retard)"))
display(pd.DataFrame(lignes).set_index("Décision"))

# Règle complète face à la vraie PAY_1, sur tous les clients du test
estimee = test["PAY_2"].copy()
estimee[retard_test] = np.where(regles["Les trois d'accord (règle retenue)"], -1, 2)
famille = lambda v: np.where(np.asarray(v) >= 2, 2, np.asarray(v))
vraie = test["PAY_1"]
lignes = []
for libelle, p in [("Règle retenue (persistance + sorties estimées)", estimee), ("Même codification qu'en août", test["PAY_2"])]:
    lignes.append({"Règle": libelle,
                   "Sain ou en retard, bonne classe (%)": round(((famille(p) >= 2) == (famille(vraie) >= 2)).mean() * 100, 1),
                   "Codification exacte (-2, -1, 0, 2 et plus) (%)": round((famille(p) == famille(vraie)).mean() * 100, 1)})
display(Markdown(f"**Test : tous les clients** ({len(test)} clients dont PAY_1 est tranchée)"))
display(pd.DataFrame(lignes).set_index("Règle"))

**Test : clients en retard en août** (543 clients, dont 103 sorties du retard)

,Sorties annoncées,Vraies sorties parmi elles,Sorties retrouvées (%),Précision des sorties annoncées (%)
Décision,,,,
MLP seul,113,90,87.4,79.6
CatBoost seul,105,88,85.4,83.8
SVM seul,109,89,86.4,81.7
Les trois d'accord (règle retenue),101,86,83.5,85.1


**Test : tous les clients** (5153 clients dont PAY_1 est tranchée)

,"Sain ou en retard, bonne classe (%)","Codification exacte (-2, -1, 0, 2 et plus) (%)"
Règle,,
Règle retenue (persistance + sorties estimées),95.8,91.0
Même codification qu'en août,94.4,89.6


**Comment lire** : le premier tableau se compare à la section 4 (validation croisée sur le train) : des chiffres proches confirment que les mesures du train tiennent sur des clients jamais vus. Le second donne la part des clients du test dont la codification de septembre est bien estimée, face à la règle naïve « même codification qu'en août » ; la règle retenue ne diffère d'elle que par les sorties estimées.

## 5 ter. Réapprentissage sur train + test

Le test a été lu une fois (section 5 bis) et la règle est confirmée : les trois modèles sont réappris sur **tous les clients dont `PAY_1` est tranchée** (train + test), pour leur donner plus d'exemples. Les seuils sont recalculés de la même façon (précision marginale de 80 %), sur les probabilités hors pli de tous ces clients (5 plis par client). Les modèles et la synthèse de `src/modeles_pay1/` sont remplacés ; la section 6 utilise ces modèles. Il n'y a plus de mesure indépendante après ce réapprentissage : les chiffres de référence restent ceux des sections 4 et 5 bis.

In [7]:
# Exemples de tous les clients dont PAY_1 est tranchée (train + test), plus les mois d'août et de juillet des clients codifiés 1
emp = pd.concat([exemples_du_mois(df.loc[connus.index], 1)]
                + [exemples_du_mois(df.loc[connus.index.union(a_trancher.index)], n) for n in (2, 3)], ignore_index=True)
emp = emp[(emp["codif_m1"] == 2) & (emp["codification_du_mois"] != 1)].reset_index(drop=True)
emp["sortie"] = (emp["codification_du_mois"] <= 0).astype(int)
X_emp, y = emp[VARIABLES], emp["sortie"].to_numpy()
sept = np.flatnonzero(emp["mois"].to_numpy() == 1)
print(f"Exemples : {len(emp)} (septembre : {len(sept)} clients en retard en août, dont {int(y[sept].sum())} sorties)")

# Seuils recalculés sur les probabilités hors pli de tous les clients, puis modèles finaux appris sur tous les exemples
plis_tout = plis_par_client(np.arange(len(emp)))
annonce_tout = {}
for nom, modele in MODELES.items():
    debut = time.perf_counter()
    p = np.full(len(emp), np.nan)
    for apprentissage, validation in plis_tout:
        p[validation] = score(clone(modele).fit(X_emp.iloc[apprentissage], y[apprentissage]), X_emp.iloc[validation])
    seuil = float(seuil_marginal(y[sept], p[sept]))
    annonce_tout[nom] = p[sept] >= seuil
    final = clone(modele).fit(X_emp, y)
    fichier = dossier / synthese["modeles"][nom]["fichier"]
    with open(fichier, "wb") as f:
        cloudpickle.dump(final, f)
    synthese["modeles"][nom]["seuil"] = seuil
    print(f"{nom} : seuil {seuil:.3f}, modèle réappris et enregistré ({round(time.perf_counter() - debut)} s)", flush=True)

synthese["date"] = datetime.now().isoformat(timespec="seconds")
synthese["exemples"] = ("clients en retard le mois précédent ; mois cibles septembre, août, juillet ; "
                        "train + test (tous les clients dont PAY_1 est tranchée), après la vérification unique sur le test")
(dossier / "synthese.json").write_text(json.dumps(synthese, ensure_ascii=False, indent=1), encoding="utf-8")

# Pour information : la règle des trois modèles sur les probabilités hors pli de tous les clients
a = sum(v.astype(int) for v in annonce_tout.values()) >= VOIX_NECESSAIRES
print(f"Sorties annoncées par les trois (hors pli) : {int(a.sum())}, dont {int(y[sept][a].sum())} vraies "
      f"({y[sept][a].mean() * 100:.1f} %) ; sorties retrouvées : {y[sept][a].sum() / y[sept].sum() * 100:.1f} %")


Exemples : 10279 (septembre : 2578 clients en retard en août, dont 447 sorties)
MLP : seuil 0.412, modèle réappris et enregistré (56 s)
CatBoost : seuil 0.610, modèle réappris et enregistré (11 s)
SVM : seuil 0.046, modèle réappris et enregistré (9 s)
Sorties annoncées par les trois (hors pli) : 400, dont 359 vraies (89.8 %) ; sorties retrouvées : 80.3 %


## 6. Construction de `PAY_1_corrigee` (le code à reprendre dans `02_01_nettoyage`)

Les trois modèles enregistrés sont rechargés et appliqués aux clients codifiés 1 en septembre et en retard en août ; la règle complète donne `PAY_1_corrigee`. Ici, la construction est faite sur `cleaned1` pour contrôle ; aucun fichier de données n'est écrit. Dans `02_01_nettoyage`, la même fonction s'applique après le niveau 1 et avant le classement au contentieux, sur les codifications corrigées à la section 2.

In [8]:
import pickle

def construire_pay1_corrigee(d, dossier_modeles):
    # d : codifications 1 d'avril à août déjà corrigées (section 2). Renvoie PAY_1_corrigee et son statut, client par client
    synthese = json.loads((dossier_modeles / "synthese.json").read_text(encoding="utf-8"))
    modeles = {}
    for nom, info in synthese["modeles"].items():
        with open(dossier_modeles / info["fichier"], "rb") as f:
            modeles[nom] = (pickle.load(f), info["seuil"])
    uns = d["PAY_1"] == 1
    sains, retard = uns & (d["PAY_2"] <= 0), uns & (d["PAY_2"] >= 2)
    assert int(uns.sum()) == int(sains.sum() + retard.sum()), "un PAY_1 = 1 n'est ni sain ni en retard en août"
    X = exemples_du_mois(d.loc[retard], 1)[synthese["variables"]]
    # Voix de chaque modèle : probabilité de sortie, ou score pour un modèle qui ne donne pas de probabilité (SVM), au-dessus de son seuil
    voix = sum(((m.predict_proba(X)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(X)) >= s).astype(int)
               for m, s in modeles.values())
    sortie = voix >= synthese["voix_necessaires"]
    corrigee = d["PAY_1"].copy()
    corrigee[sains] = d.loc[sains, "PAY_2"]
    corrigee[retard] = np.where(sortie, -1, 2)
    statut = pd.Series("PAY_1 tranchée par la banque", index=d.index)
    statut[sains] = "1, sain en août : codification d'août"
    statut[retard] = np.where(sortie, "1, en retard en août : sortie (les trois modèles)",
                              np.where(voix >= 1, "1, en retard en août : retard maintenu (sortie annoncée par une partie des modèles)",
                                       "1, en retard en août : retard maintenu (aucun modèle)"))
    return corrigee, statut

pay1_corrigee, statut = construire_pay1_corrigee(df, dossier)
assert not (pay1_corrigee == 1).any()
display(pd.crosstab(statut.rename("Statut"), pay1_corrigee.rename("PAY_1_corrigee"), margins=True, margins_name="Total"))

PAY_1_corrigee,-2,-1,0,2,3,4,5,6,7,8,Total
Statut,,,,,,,,,,,
"1, en retard en août : retard maintenu (aucun modèle)",0,0,0,1530,0,0,0,0,0,0,1530
"1, en retard en août : retard maintenu (sortie annoncée par une partie des modèles)",0,0,0,93,0,0,0,0,0,0,93
"1, en retard en août : sortie (les trois modèles)",0,199,0,0,0,0,0,0,0,0,199
"1, sain en août : codification d'août",640,617,7,0,0,0,0,0,0,0,1264
PAY_1 tranchée par la banque,2372,5653,14620,2658,322,76,26,11,9,18,25765
Total,3012,6469,14627,4281,322,76,26,11,9,18,28851


**Comment lire** : une ligne par statut, une colonne par valeur de `PAY_1_corrigee`. Les clients dont la banque a tranché `PAY_1` gardent leur codification ; les codifications 1 sont réparties selon la règle. Le statut sert à lire la colonne ; il n'est pas une variable des modèles.